# Figure 2 — Source Data Export

**Figure 2** shows an example PIT (posterior inferotemporal) neuron evolution experiment (Exp 155),
demonstrating the evolution trajectory and population PSTH dynamics across generations.

## Data requirements

> **Raw neural recordings are required to run this notebook.**
> The `.mat`/`.pkl` files are not distributed with the repository.
> Set the `MATROOT` environment variable to the directory containing
> `Both_BigGAN_FC6_Evol_Stats.pkl` and `Both_BigGAN_FC6_Evol_Stats_expsep.pkl`
> before launching Jupyter, or assign the path directly to `matroot` in the
> cell below.
>
> ```bash
> export MATROOT=/path/to/Mat_Statistics
> ```

The exported source data files are written to `../source_data/Fig2/`.

In [ ]:
import os, sys, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ".."))
from neuro_data_analysis.neural_data_lib import (
    load_neural_data,
    extract_all_evol_trajectory_psth,
    pad_psth_traj,
    get_expstr,
    extract_natref_activation_array,
    extract_evol_activation_array,
)
from neuro_data_analysis.neural_data_utils import get_all_masks

In [ ]:
# Set the path to the raw neural recording data.
# Override by setting the MATROOT environment variable before launching Jupyter.
matroot = os.environ.get("MATROOT", "")

BFEStats_merge, BFEStats = load_neural_data()

# Build experiment-level metadata and standard inclusion masks
from neuro_data_analysis.neural_data_lib import extract_all_evol_trajectory_dyna, pad_resp_traj
resp_col, meta_df = extract_all_evol_trajectory_dyna(BFEStats)
Amsk, Bmsk, V1msk, V4msk, ITmsk, length_msk, spc_msk, sucsmsk, \
    bsl_unstable_msk, bsl_stable_msk, validmsk = get_all_masks(meta_df)

print(f"Total experiments: {len(BFEStats)}")
print(f"Valid experiments: {validmsk.sum()}")

## Figure 2D: PSTH stack export

Extract the per-block mean and SEM PSTH time-courses for experiment 155
(example PIT neuron, DeePSim thread 0 and BigGAN thread 1).
The PSTH arrays are offset-corrected using the pre-stimulus baseline window (0–45 ms).

In [ ]:
outdir = "../source_data/Fig2"
os.makedirs(outdir, exist_ok=True)

Expi = 155
psth_col, meta_psth = extract_all_evol_trajectory_psth(BFEStats)

# psth_col[Expi] shape: (n_blocks, n_threads, n_timebins)
psth_exp = psth_col[Expi]         # list of per-block arrays
psth_padded = pad_psth_traj(psth_exp)   # (n_blocks, n_threads, T)

# Baseline offset: subtract mean of pre-stimulus window (0–45 ms)
bsl_wdw = range(0, 45)
bsl_offset = psth_padded[:, :, bsl_wdw].mean(axis=2, keepdims=True)
psth_offset = psth_padded - bsl_offset

# Thread 0 = DeePSim (fc6), Thread 1 = BigGAN
for thr_idx, space_name in enumerate(["DeePSim", "BigGAN"]):
    thr_data = psth_offset[:, thr_idx, :]           # (n_blocks, T)
    thr_mean = thr_data.mean(axis=0)                # (T,)
    thr_sem  = thr_data.std(axis=0) / np.sqrt(thr_data.shape[0])

    pd.DataFrame(thr_mean[np.newaxis, :]).to_csv(
        join(outdir, f"Figure2D_src_psth_Exp{Expi}_{space_name}_mean.csv"), index=False)
    pd.DataFrame(thr_sem[np.newaxis, :]).to_csv(
        join(outdir, f"Figure2D_src_psth_Exp{Expi}_{space_name}_sem.csv"), index=False)

# Save full padded array as pickle
pickle.dump({"psth_offset": psth_offset, "Expi": Expi},
            open(join(outdir, f"Figure2D_src_psth_Exp{Expi}.pkl"), "wb"))

# Metadata text
expstr = get_expstr(BFEStats, Expi)
meta_txt = f"{expstr}\nPSTH shape (n_blocks, n_threads, T): {psth_padded.shape}\n"
with open(join(outdir, f"Figure2D_src_psth_Exp{Expi}_meta.txt"), "w") as f:
    f.write(meta_txt)

print("Figure 2D source data saved to", outdir)

## Figure 2B: Evolution trajectory export

Extract per-block mean firing rates for the evolution (evol) and natural-image
reference (natref) stimuli for both GAN threads of experiment 155.

In [ ]:
S = BFEStats[Expi - 1]
result = {}

for thr_idx, space_name in enumerate(["DeePSim", "BigGAN"]):
    resp_arr, bsl_arr, gen_arr, _, _, _ = extract_evol_activation_array(
        S, thread=thr_idx, rsp_wdw=range(50, 200), bsl_wdw=range(0, 45)
    )
    resp_mean = np.array([r.mean() for r in resp_arr])
    bsl_mean  = np.array([b.mean() for b in bsl_arr])
    gen_mean  = np.array([g.mean() for g in gen_arr])

    evol_df = pd.DataFrame({
        "generation": gen_mean,
        "resp_mean": resp_mean,
        "bsl_mean": bsl_mean,
    })
    evol_df.to_csv(join(outdir, f"Figure2B_src_resp_Exp{Expi}_{space_name}_thread_evol.csv"), index=False)
    result[f"{space_name}_evol"] = evol_df

    nat_resp_arr, nat_bsl_arr, nat_gen_arr = extract_natref_activation_array(
        S, thread=thr_idx, rsp_wdw=range(50, 200), bsl_wdw=range(0, 45)
    )
    natref_df = pd.DataFrame({
        "generation": nat_gen_arr,
        "resp_mean": nat_resp_arr,
        "bsl_mean": nat_bsl_arr,
    })
    natref_df.to_csv(join(outdir, f"Figure2B_src_resp_Exp{Expi}_{space_name}_thread_natref.csv"), index=False)
    result[f"{space_name}_natref"] = natref_df

pickle.dump(result, open(join(outdir, f"Figure2B_src_resp_Exp{Expi}.pkl"), "wb"))
print("Figure 2B source data saved to", outdir)